In [1]:
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, classification_report
import joblib

In [4]:
# ==========================
# 1. Chargement des données
# ==========================

DATA_PATH = Path("SpotifyAudioFeaturesApril2019.csv")  # adapte si besoin

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("Colonnes:", df.columns.tolist())

# Création du label
df["is_hit"] = (df["popularity"] >= 60).astype(int)

print(df["is_hit"].value_counts(normalize=True))

Shape: (130663, 17)
Colonnes: ['artist_name', 'track_id', 'track_name', 'acousticness', 'danceability', 'duration_ms', 'energy', 'instrumentalness', 'key', 'liveness', 'loudness', 'mode', 'speechiness', 'tempo', 'time_signature', 'valence', 'popularity']
is_hit
0    0.944919
1    0.055081
Name: proportion, dtype: float64


In [5]:
# Features audio
FEATURE_COLS = [
    "acousticness",
    "danceability",
    "duration_ms",
    "energy",
    "instrumentalness",
    "key",
    "liveness",
    "loudness",
    "mode",
    "speechiness",
    "tempo",
    "time_signature",
    "valence",
]

TARGET_COL = "is_hit"

# On enlève les lignes avec NaN sur les features (il ne devrait pas y en avoir, mais par sécurité)
df_model = df.dropna(subset=FEATURE_COLS + [TARGET_COL]).copy()

X = df_model[FEATURE_COLS]
y = df_model[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,  # important car la classe "hit" est rare
)

print("Train size:", X_train.shape, "Test size:", X_test.shape)


Train size: (104530, 13) Test size: (26133, 13)


In [6]:
from sklearn.utils import class_weight

pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "clf",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",  # important !
            ),
        ),
    ]
)

pipeline.fit(X_train, y_train)

# Probabilités pour la classe 1 (hit)
proba_test = pipeline.predict_proba(X_test)[:, 1]
y_pred = (proba_test >= 0.5).astype(int)

roc = roc_auc_score(y_test, proba_test)
pr = average_precision_score(y_test, proba_test)
f1 = f1_score(y_test, y_pred)

print(f"ROC-AUC: {roc:.3f}")
print(f"PR-AUC:  {pr:.3f}")
print(f"F1:      {f1:.3f}")
print("\nClassification report:\n", classification_report(y_test, y_pred))


ROC-AUC: 0.710
PR-AUC:  0.111
F1:      0.165

Classification report:
               precision    recall  f1-score   support

           0       0.98      0.56      0.71     24694
           1       0.09      0.77      0.17      1439

    accuracy                           0.57     26133
   macro avg       0.53      0.66      0.44     26133
weighted avg       0.93      0.57      0.68     26133



In [7]:
import numpy as np

clf = pipeline.named_steps["clf"]
coefs = clf.coef_[0]

feat_importance = sorted(
    zip(FEATURE_COLS, coefs),
    key=lambda x: abs(x[1]),
    reverse=True,
)

print("Features les plus influentes (en valeur absolue des coefficients) :")
for name, coef in feat_importance:
    print(f"{name:15s} -> coef = {coef:.3f}")



Features les plus influentes (en valeur absolue des coefficients) :
loudness        -> coef = 0.880
energy          -> coef = -0.471
instrumentalness -> coef = -0.308
danceability    -> coef = 0.267
duration_ms     -> coef = -0.256
valence         -> coef = -0.180
liveness        -> coef = -0.102
acousticness    -> coef = 0.071
speechiness     -> coef = -0.051
time_signature  -> coef = 0.051
mode            -> coef = -0.026
tempo           -> coef = 0.017
key             -> coef = -0.008


In [8]:
import pandas as pd

# Reconstituer un DataFrame train complet pour faciliter le sampling
train_df = X_train.copy()
train_df[TARGET_COL] = y_train.values

hits = train_df[train_df[TARGET_COL] == 1]
non_hits = train_df[train_df[TARGET_COL] == 0]

print("Hits train:", len(hits))
print("Non-hits train:", len(non_hits))

# Choisir un ratio, par ex : 1 hit -> 4 non-hits
ratio = 4
n_non_hits_keep = min(len(non_hits), len(hits) * ratio)

non_hits_under = non_hits.sample(n=n_non_hits_keep, random_state=42)

train_bal = pd.concat([hits, non_hits_under], axis=0).sample(frac=1, random_state=42)  # shuffle

X_train_bal = train_bal[FEATURE_COLS]
y_train_bal = train_bal[TARGET_COL]

print("Nouvelle distrib train :")
print(y_train_bal.value_counts(normalize=True))


Hits train: 5758
Non-hits train: 98772
Nouvelle distrib train :
is_hit
0    0.8
1    0.2
Name: proportion, dtype: float64


In [9]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2000, class_weight=None)),  # class_weight=None car on a déjà rééquilibré
    ]
)

pipeline.fit(X_train_bal, y_train_bal)

# Évaluation sur le test original (déséquilibré)
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, classification_report

proba_test = pipeline.predict_proba(X_test)[:, 1]
y_pred = (proba_test >= 0.5).astype(int)

roc = roc_auc_score(y_test, proba_test)
pr = average_precision_score(y_test, proba_test)
f1 = f1_score(y_test, y_pred)

print(f"ROC-AUC: {roc:.3f}")
print(f"PR-AUC:  {pr:.3f}")
print(f"F1:      {f1:.3f}")
print("\nClassification report:\n", classification_report(y_test, y_pred))


ROC-AUC: 0.714
PR-AUC:  0.114
F1:      0.011

Classification report:
               precision    recall  f1-score   support

           0       0.95      1.00      0.97     24694
           1       0.25      0.01      0.01      1439

    accuracy                           0.94     26133
   macro avg       0.60      0.50      0.49     26133
weighted avg       0.91      0.94      0.92     26133



In [10]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
print(cm)

import numpy as np

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print("Confusion matrix :")
print(cm)
print()
print(f"TN (0 bien prédit) : {tn}")
print(f"FP (faux hits)     : {fp}")
print(f"FN (hits ratés)    : {fn}")
print(f"TP (hits trouvés)  : {tp}")


[[24670    24]
 [ 1431     8]]
Confusion matrix :
[[24670    24]
 [ 1431     8]]

TN (0 bien prédit) : 24670
FP (faux hits)     : 24
FN (hits ratés)    : 1431
TP (hits trouvés)  : 8


In [13]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
)

import joblib


# ==========================
# 0. CONFIG
# ==========================

DATA_PATH = Path("SpotifyAudioFeaturesApril2019.csv")  # adapte si besoin
MODEL_PATH = Path("hitscope_model.pkl")
THRESHOLD_PATH = Path("hitscope_threshold.txt")

# Seuil de popularité pour appeler un titre "hit"
POPULARITY_THRESHOLD = 60

# Ratio d'undersampling : 1 hit pour RATIO non-hits dans le train
UNDERSAMPLING_RATIO = 4.0  # 1 hit : 4 non-hits


# ==========================
# 1. Chargement & label
# ==========================

print("📥 Chargement des données…")
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("Colonnes:", df.columns.tolist())

# Création du label binaire
df["is_hit"] = (df["popularity"] >= POPULARITY_THRESHOLD).astype(int)
print("\nDistribution de is_hit :")
print(df["is_hit"].value_counts(normalize=True))

# Features audio (numériques)
FEATURE_COLS = [
    "acousticness",
    "danceability",
    "duration_ms",
    "energy",
    "instrumentalness",
    "key",
    "liveness",
    "loudness",
    "mode",
    "speechiness",
    "tempo",
    "time_signature",
    "valence",
]
TARGET_COL = "is_hit"

# On supprime les lignes avec NaN sur features/target, par sécurité
df_model = df.dropna(subset=FEATURE_COLS + [TARGET_COL]).copy()

X = df_model[FEATURE_COLS]
y = df_model[TARGET_COL]


# ==========================
# 2. Split train / test
# ==========================

print("\n✂️ Split train / test (stratifié)…")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Taille train:", X_train.shape, "Taille test:", X_test.shape)
print("Distribution is_hit train:")
print(y_train.value_counts(normalize=True))
print("Distribution is_hit test:")
print(y_test.value_counts(normalize=True))


# ==========================
# 3. Undersampling des non-hits dans le train
# ==========================

def undersample_train(X_train, y_train, ratio: float = 4.0):
    """
    Conserve tous les hits (classe 1) et échantillonne les non-hits (classe 0)
    pour obtenir environ ratio non-hits par hit.
    """
    train_df = X_train.copy()
    train_df[TARGET_COL] = y_train.values

    hits = train_df[train_df[TARGET_COL] == 1]
    non_hits = train_df[train_df[TARGET_COL] == 0]

    n_hits = len(hits)
    n_non_hits_keep = min(len(non_hits), int(n_hits * ratio))

    non_hits_under = non_hits.sample(n=n_non_hits_keep, random_state=42)

    train_bal = pd.concat([hits, non_hits_under], axis=0).sample(
        frac=1, random_state=42
    )  # shuffle

    X_train_bal = train_bal[FEATURE_COLS]
    y_train_bal = train_bal[TARGET_COL]

    return X_train_bal, y_train_bal


print("\n⚖️ Undersampling du train…")
X_train_bal, y_train_bal = undersample_train(X_train, y_train, ratio=UNDERSAMPLING_RATIO)

print("Distribution is_hit dans le train rééquilibré :")
print(y_train_bal.value_counts(normalize=True))


# ==========================
# 4. Pipeline ML (Logistic Regression)
# ==========================

print("\n🧠 Entraînement du modèle (Logistic Regression)…")

pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "clf",
            LogisticRegression(
                max_iter=2000,
                class_weight=None,  # on a déjà rééquilibré les données
            ),
        ),
    ]
)

pipeline.fit(X_train_bal, y_train_bal)


# ==========================
# 5. Évaluation : ROC-AUC, PR-AUC, F1, etc.
# ==========================

print("\n📊 Évaluation sur le test (distribution réelle)…")

proba_test = pipeline.predict_proba(X_test)[:, 1]

roc = roc_auc_score(y_test, proba_test)
pr_auc = average_precision_score(y_test, proba_test)

print(f"ROC-AUC (probas): {roc:.3f}")
print(f"PR-AUC  (probas): {pr_auc:.3f}")


# ==========================
# 6. Comparaison seuil 0.5 vs meilleur seuil F1
# ==========================

def evaluate_at_threshold(thresh: float, y_true, y_proba):
    y_pred = (y_proba >= thresh).astype(int)
    f1 = f1_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)
    return f1, prec, rec, cm, y_pred


print("\n🔎 Évaluation au seuil 0.5 :")
f1_05, prec_05, rec_05, cm_05, y_pred_05 = evaluate_at_threshold(0.5, y_test, proba_test)

print(f"Threshold = 0.5")
print(f"  Precision (classe 1) : {prec_05:.3f}")
print(f"  Recall    (classe 1) : {rec_05:.3f}")
print(f"  F1        (classe 1) : {f1_05:.3f}")
print("  Matrice de confusion :")
tn, fp, fn, tp = cm_05.ravel()
print(f"    TN : {tn}")
print(f"    FP : {fp}")
print(f"    FN : {fn}")
print(f"    TP : {tp}")


# Recherche du meilleur seuil en termes de F1
print("\n🔍 Recherche du meilleur seuil (max F1)…")
best_thresh = 0.5
best_f1 = 0.0
best_prec = 0.0
best_rec = 0.0
best_cm = None
best_y_pred = None

for t in np.linspace(0.01, 0.99, 99):
    f1_t, prec_t, rec_t, cm_t, y_pred_t = evaluate_at_threshold(t, y_test, proba_test)
    if f1_t > best_f1:
        best_f1 = f1_t
        best_prec = prec_t
        best_rec = rec_t
        best_thresh = t
        best_cm = cm_t
        best_y_pred = y_pred_t

print(f"✅ Meilleur seuil trouvé : {best_thresh:.3f}")
print(f"  Precision (classe 1) : {best_prec:.3f}")
print(f"  Recall    (classe 1) : {best_rec:.3f}")
print(f"  F1        (classe 1) : {best_f1:.3f}")

tn, fp, fn, tp = best_cm.ravel()
print("  Matrice de confusion au meilleur seuil :")
print(f"    TN : {tn}")
print(f"    FP : {fp}")
print(f"    FN : {fn}")
print(f"    TP : {tp}")

print("\nClassification report (meilleur seuil) :")
print(classification_report(y_test, best_y_pred))


# ==========================
# 7. Sauvegarde du modèle & seuil
# ==========================

print("\n💾 Sauvegarde du modèle et du seuil…")
joblib.dump(pipeline, MODEL_PATH)
with open(THRESHOLD_PATH, "w") as f:
    f.write(str(best_thresh))

print(f"Modèle sauvegardé dans : {MODEL_PATH}")
print(f"Seuil optimal sauvegardé dans : {THRESHOLD_PATH}")
print("\nTerminé ✅")


📥 Chargement des données…
Shape: (130663, 17)
Colonnes: ['artist_name', 'track_id', 'track_name', 'acousticness', 'danceability', 'duration_ms', 'energy', 'instrumentalness', 'key', 'liveness', 'loudness', 'mode', 'speechiness', 'tempo', 'time_signature', 'valence', 'popularity']

Distribution de is_hit :
is_hit
0    0.944919
1    0.055081
Name: proportion, dtype: float64

✂️ Split train / test (stratifié)…
Taille train: (104530, 13) Taille test: (26133, 13)
Distribution is_hit train:
is_hit
0    0.944915
1    0.055085
Name: proportion, dtype: float64
Distribution is_hit test:
is_hit
0    0.944936
1    0.055064
Name: proportion, dtype: float64

⚖️ Undersampling du train…
Distribution is_hit dans le train rééquilibré :
is_hit
0    0.8
1    0.2
Name: proportion, dtype: float64

🧠 Entraînement du modèle (Logistic Regression)…

📊 Évaluation sur le test (distribution réelle)…
ROC-AUC (probas): 0.714
PR-AUC  (probas): 0.114

🔎 Évaluation au seuil 0.5 :
Threshold = 0.5
  Precision (classe 1) 

In [11]:
# ==========================
# 8. Modèle RandomForest (plus puissant)
# ==========================

from sklearn.ensemble import RandomForestClassifier

print("\n🌲 Entraînement d'un RandomForest…")

rf_pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),  # pas indispensable pour RF mais homogène
        (
            "clf",
            RandomForestClassifier(
                n_estimators=500,
                max_depth=None,
                min_samples_split=2,
                min_samples_leaf=1,
                n_jobs=-1,
                random_state=42,
                class_weight=None,  # on a déjà undersample
            ),
        ),
    ]
)

rf_pipeline.fit(X_train_bal, y_train_bal)

print("\n📊 Évaluation RandomForest sur le test (distribution réelle)…")
rf_proba_test = rf_pipeline.predict_proba(X_test)[:, 1]

rf_roc = roc_auc_score(y_test, rf_proba_test)
rf_pr_auc = average_precision_score(y_test, rf_proba_test)

print(f"[RF] ROC-AUC (probas): {rf_roc:.3f}")
print(f"[RF] PR-AUC  (probas): {rf_pr_auc:.3f}")


# --- Seuil 0.5 pour RF
print("\n🔎 [RF] Évaluation au seuil 0.5 :")
rf_f1_05, rf_prec_05, rf_rec_05, rf_cm_05, rf_y_pred_05 = evaluate_at_threshold(
    0.5, y_test, rf_proba_test
)

print(f"[RF] Threshold = 0.5")
print(f"  Precision (classe 1) : {rf_prec_05:.3f}")
print(f"  Recall    (classe 1) : {rf_rec_05:.3f}")
print(f"  F1        (classe 1) : {rf_f1_05:.3f}")
print("  Matrice de confusion :")
tn, fp, fn, tp = rf_cm_05.ravel()
print(f"    TN : {tn}")
print(f"    FP : {fp}")
print(f"    FN : {fn}")
print(f"    TP : {tp}")


# --- Recherche meilleur seuil pour RF
print("\n🔍 [RF] Recherche du meilleur seuil (max F1)…")
rf_best_thresh = 0.5
rf_best_f1 = 0.0
rf_best_prec = 0.0
rf_best_rec = 0.0
rf_best_cm = None
rf_best_y_pred = None

for t in np.linspace(0.01, 0.99, 99):
    f1_t, prec_t, rec_t, cm_t, y_pred_t = evaluate_at_threshold(
        t, y_test, rf_proba_test
    )
    if f1_t > rf_best_f1:
        rf_best_f1 = f1_t
        rf_best_prec = prec_t
        rf_best_rec = rec_t
        rf_best_thresh = t
        rf_best_cm = cm_t
        rf_best_y_pred = y_pred_t

print(f"✅ [RF] Meilleur seuil : {rf_best_thresh:.3f}")
print(f"  Precision (classe 1) : {rf_best_prec:.3f}")
print(f"  Recall    (classe 1) : {rf_best_rec:.3f}")
print(f"  F1        (classe 1) : {rf_best_f1:.3f}")

tn, fp, fn, tp = rf_best_cm.ravel()
print("  Matrice de confusion au meilleur seuil [RF] :")
print(f"    TN : {tn}")
print(f"    FP : {fp}")
print(f"    FN : {fn}")
print(f"    TP : {tp}")

print("\n[RF] Classification report (meilleur seuil) :")
print(classification_report(y_test, rf_best_y_pred))

# (optionnel) Sauvegarde du modèle RF + seuil optimal
RF_MODEL_PATH = Path("hitscope_model_rf.pkl")
RF_THRESHOLD_PATH = Path("hitscope_threshold_rf.txt")

joblib.dump(rf_pipeline, RF_MODEL_PATH)
with open(RF_THRESHOLD_PATH, "w") as f:
    f.write(str(rf_best_thresh))

print(f"\n💾 Modèle RF sauvegardé dans : {RF_MODEL_PATH}")
print(f"💾 Seuil RF optimal sauvegardé dans : {RF_THRESHOLD_PATH}")



🌲 Entraînement d'un RandomForest…

📊 Évaluation RandomForest sur le test (distribution réelle)…
[RF] ROC-AUC (probas): 0.811
[RF] PR-AUC  (probas): 0.090

🔎 [RF] Évaluation au seuil 0.5 :
[RF] Threshold = 0.5
  Precision (classe 1) : 0.097
  Recall    (classe 1) : 0.213
  F1        (classe 1) : 0.133
  Matrice de confusion :
    TN : 24927
    FP : 802
    FN : 318
    TP : 86

🔍 [RF] Recherche du meilleur seuil (max F1)…
✅ [RF] Meilleur seuil : 0.510
  Precision (classe 1) : 0.103
  Recall    (classe 1) : 0.198
  F1        (classe 1) : 0.135
  Matrice de confusion au meilleur seuil [RF] :
    TN : 25030
    FP : 699
    FN : 324
    TP : 80

[RF] Classification report (meilleur seuil) :
              precision    recall  f1-score   support

           0       0.99      0.97      0.98     25729
           1       0.10      0.20      0.14       404

    accuracy                           0.96     26133
   macro avg       0.54      0.59      0.56     26133
weighted avg       0.97      0

In [13]:
# ==========================
# 9. Modèle XGBoost (optionnel)
# ==========================

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("\n⚠️ xgboost n'est pas installé. `pip install xgboost` pour activer cette partie.")

if HAS_XGB:
    print("\n🔥 Entraînement d'un XGBoost…")

    xgb_pipeline = Pipeline(
        steps=[
            ("scaler", StandardScaler()),  # pas strictement nécessaire, mais ok
            (
                "clf",
                XGBClassifier(
                    n_estimators=400,
                    max_depth=6,
                    learning_rate=0.05,
                    subsample=0.9,
                    colsample_bytree=0.9,
                    objective="binary:logistic",
                    eval_metric="logloss",
                    n_jobs=4,
                    random_state=42,
                    scale_pos_weight=1.0,  # tu peux ajuster si tu ne sous-échantillonnais pas
                ),
            ),
        ]
    )

    xgb_pipeline.fit(X_train_bal, y_train_bal)

    print("\n📊 Évaluation XGBoost sur le test (distribution réelle)…")
    xgb_proba_test = xgb_pipeline.predict_proba(X_test)[:, 1]

    xgb_roc = roc_auc_score(y_test, xgb_proba_test)
    xgb_pr_auc = average_precision_score(y_test, xgb_proba_test)

    print(f"[XGB] ROC-AUC (probas): {xgb_roc:.3f}")
    print(f"[XGB] PR-AUC  (probas): {xgb_pr_auc:.3f}")

    # --- Seuil 0.5
    print("\n🔎 [XGB] Évaluation au seuil 0.5 :")
    xgb_f1_05, xgb_prec_05, xgb_rec_05, xgb_cm_05, xgb_y_pred_05 = evaluate_at_threshold(
        0.5, y_test, xgb_proba_test
    )

    print(f"[XGB] Threshold = 0.5")
    print(f"  Precision (classe 1) : {xgb_prec_05:.3f}")
    print(f"  Recall    (classe 1) : {xgb_rec_05:.3f}")
    print(f"  F1        (classe 1) : {xgb_f1_05:.3f}")
    print("  Matrice de confusion :")
    tn, fp, fn, tp = xgb_cm_05.ravel()
    print(f"    TN : {tn}")
    print(f"    FP : {fp}")
    print(f"    FN : {fn}")
    print(f"    TP : {tp}")

    # --- Recherche du meilleur seuil pour XGB
    print("\n🔍 [XGB] Recherche du meilleur seuil (max F1)…")
    xgb_best_thresh = 0.5
    xgb_best_f1 = 0.0
    xgb_best_prec = 0.0
    xgb_best_rec = 0.0
    xgb_best_cm = None
    xgb_best_y_pred = None

    for t in np.linspace(0.01, 0.99, 99):
        f1_t, prec_t, rec_t, cm_t, y_pred_t = evaluate_at_threshold(
            t, y_test, xgb_proba_test
        )
        if f1_t > xgb_best_f1:
            xgb_best_f1 = f1_t
            xgb_best_prec = prec_t
            xgb_best_rec = rec_t
            xgb_best_thresh = t
            xgb_best_cm = cm_t
            xgb_best_y_pred = y_pred_t

    print(f"✅ [XGB] Meilleur seuil : {xgb_best_thresh:.3f}")
    print(f"  Precision (classe 1) : {xgb_best_prec:.3f}")
    print(f"  Recall    (classe 1) : {xgb_best_rec:.3f}")
    print(f"  F1        (classe 1) : {xgb_best_f1:.3f}")

    tn, fp, fn, tp = xgb_best_cm.ravel()
    print("  Matrice de confusion au meilleur seuil [XGB] :")
    print(f"    TN : {tn}")
    print(f"    FP : {fp}")
    print(f"    FN : {fn}")
    print(f"    TP : {tp}")

    print("\n[XGB] Classification report (meilleur seuil) :")
    print(classification_report(y_test, xgb_best_y_pred))

    # Sauvegarde
    XGB_MODEL_PATH = Path("hitscope_model_xgb.pkl")
    XGB_THRESHOLD_PATH = Path("hitscope_threshold_xgb.txt")

    joblib.dump(xgb_pipeline, XGB_MODEL_PATH)
    with open(XGB_THRESHOLD_PATH, "w") as f:
        f.write(str(xgb_best_thresh))

    print(f"\n💾 Modèle XGB sauvegardé dans : {XGB_MODEL_PATH}")
    print(f"💾 Seuil XGB optimal sauvegardé dans : {XGB_THRESHOLD_PATH}")



🔥 Entraînement d'un XGBoost…

📊 Évaluation XGBoost sur le test (distribution réelle)…
[XGB] ROC-AUC (probas): 0.822
[XGB] PR-AUC  (probas): 0.072

🔎 [XGB] Évaluation au seuil 0.5 :
[XGB] Threshold = 0.5
  Precision (classe 1) : 0.071
  Recall    (classe 1) : 0.295
  F1        (classe 1) : 0.114
  Matrice de confusion :
    TN : 24161
    FP : 1568
    FN : 285
    TP : 119

🔍 [XGB] Recherche du meilleur seuil (max F1)…
✅ [XGB] Meilleur seuil : 0.540
  Precision (classe 1) : 0.084
  Recall    (classe 1) : 0.260
  F1        (classe 1) : 0.127
  Matrice de confusion au meilleur seuil [XGB] :
    TN : 24582
    FP : 1147
    FN : 299
    TP : 105

[XGB] Classification report (meilleur seuil) :
              precision    recall  f1-score   support

           0       0.99      0.96      0.97     25729
           1       0.08      0.26      0.13       404

    accuracy                           0.94     26133
   macro avg       0.54      0.61      0.55     26133
weighted avg       0.97     